In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR= "/content/drive/MyDrive/NLP/ngram_model"



In [ ]:
import pickle
from collections import defaultdict

In [ ]:
def load_ngram_counts(ngram_path):
    """Load n-gram counts dict from pickle."""
    with open(ngram_path, "rb") as f:
        # return defaultdict(int, pickle.load(f))
        return pickle.load(f)

# Load n-gram counts
uni_c    = load_ngram_counts(f"{MODEL_DIR}/final_1gram_counts.pkl")
bi_c     = load_ngram_counts(f"{MODEL_DIR}/final_2gram_counts.pkl")
tri_c    = load_ngram_counts(f"{MODEL_DIR}/final_3gram_counts.pkl")
quad_c = load_ngram_counts(f"{MODEL_DIR}/final_4gram_counts.pkl")

In [ ]:
i=0

for ngram in bi_c:
  i=i+1
  *history, word = ngram[:-1],ngram[-1]
  print("ngram:",ngram,"history ",ngram[:-1]," word ",ngram[-1])
  if i>3:  break

ngram: ('<s>', '"') history  ('<s>',)  word  "
ngram: ('"', 'content') history  ('"',)  word  content
ngram: ('content', '"') history  ('content',)  word  "
ngram: ('"', ':') history  ('"',)  word  :


In [ ]:

# Discount parameter
D = 0.75

# Continuation counts
def continuation_counts(ngram_counts):
    cont = defaultdict(set)
    for ngram in ngram_counts:
        *history, word = ngram[:-1],ngram[-1]
        cont[word].add(tuple(history))
    return {w: len(h) for w, h in cont.items()}

cont_uni = continuation_counts(bi_c)
cont_bi  = continuation_counts(tri_c)
cont_tri = continuation_counts(quad_c)

total_cont_uni = sum(cont_uni.values())

def kn_uni(w):
    return cont_uni.get(w, 0) / total_cont_uni if total_cont_uni > 0 else 1e-7

def kn_bi(w3, w4):
    bi_key = (w3, w4)
    uni_key = w3
    bi_count = bi_c.get(bi_key, 0)
    uni_count = uni_c.get(uni_key, 0)

    if uni_count > 0:
        lambda_bi = (D / uni_count) * len([w for w in bi_c if w[0] == w3])
        return max(bi_count - D, 0) / uni_count + lambda_bi * kn_uni(w4)
    else:
        return kn_uni(w4)


def kn_tri(w2, w3, w4):
    tri_key = (w2, w3, w4)
    bi_key = (w2, w3)
    tri_count = tri_c.get(bi_key, 0)
    trigram_count = tri_c.get(tri_key, 0)

    if tri_count > 0:
        lambda_tri = (D / tri_count) * len([w for w in tri_c if w[:2] == bi_key])
        return max(trigram_count - D, 0) / tri_count + lambda_tri * kn_bi(w3, w4)
    else:
        return kn_bi(w3, w4)

def kn_quad(w1, w2, w3, w4):
    quad = (w1, w2, w3, w4)
    tri_key = (w1, w2, w3)
    tri_count = tri_c.get(tri_key, 0)
    quad_count = quad_c.get(quad, 0)

    if tri_count > 0:
        lambda_quad = (D / tri_count) * len([w for w in quad_c if w[:3] == tri_key])
        prob_quad = max(quad_count - D, 0) / tri_count + lambda_quad * kn_tri(w2, w3, w4)
        return prob_quad
    else:
        return kn_tri(w2, w3, w4)

In [ ]:
smoothed_probs = {}

for quad in quad_c:
    w1, w2, w3, w4 = quad
    smoothed_probs[quad] = kn_quad(w1, w2, w3, w4)


In [ ]:
output_path = "/content/drive/MyDrive/NLP/Assignment6" + "quad_kn_smooth.pkl"

with open(output_path, "wb") as f:
    pickle.dump(smoothed_probs, f)

print(" Smoothed quadgram probabilities saved at:", output_path)
